# 🏀 Basketball Player Detection & Tracking System
## LOCAL VERSION — YOLOv11m + ByteTrack
### Big Vision Internship Assignment

---

## 🎯 WHAT THIS PROJECT DOES

This notebook builds a **complete end-to-end computer vision system** that:
1. **Detects** basketball players and the ball in every frame of a game video
2. **Tracks** each player across frames with a consistent ID number
3. **Visualizes** movement trails, heatmaps, and analytics dashboards
4. **Serves** an interactive web demo (drag-drop video → tracked output)

---

## 🧠 ALGORITHM CHOICES — INTERVIEW ANSWERS

### Detection Model: YOLOv11m
| Question | Answer |
|---|---|
| **What is it?** | You Only Look Once v11, Medium variant — a real-time single-stage object detector |
| **Pre-trained?** | ✅ YES — COCO dataset (Microsoft, 118,000 images, 80 classes including `person`) |
| **Source** | https://github.com/ultralytics/ultralytics — official Ultralytics release |
| **Why not Faster R-CNN?** | Too slow: 5–10 FPS. Basketball needs 25+ FPS for real-time tracking |
| **Why not DETR?** | Needs 100k+ images to converge. We have ~5000 basketball images |
| **Why medium (m)?** | Best accuracy/speed balance on GPU. Fits in 15GB VRAM with batch=16 |
| **Approach** | Fine-tuning: starts from COCO weights, adapts to basketball in ~90 min |

### Tracking Algorithm: ByteTrack
| Question | Answer |
|---|---|
| **What is it?** | Multi-object tracker using Kalman filter + two-stage detection association |
| **Pre-trained?** | ✅ YES — built-in Kalman filter motion model (`bytetrack.yaml`, Ultralytics) |
| **Why not SORT?** | SORT drops low-confidence detections → loses ID when players cross each other |
| **Why not DeepSORT?** | Uses appearance CNN → same jersey = confused. Also slower (30 FPS vs 171 FPS) |
| **Key innovation** | Uses ALL detections (high + low confidence). Low-conf boxes keep occluded players tracked |
| **Performance** | MOTA=77.3%, 558 ID switches vs SORT's 1000+, runs at 171 FPS |

---

## ⚠️ LOCAL SETUP — DO THIS FIRST

### Step 1: Install Python 3.10+
Download from https://python.org

### Step 2: Create virtual environment (recommended)
```bash
python -m venv basketball_env
basketball_env\Scripts\activate   # Windows
source basketball_env/bin/activate  # Mac/Linux
```

### Step 3: Install all dependencies
```bash
pip install ultralytics==8.3.40 roboflow supervision lap seaborn gradio
pip install opencv-python matplotlib pandas torch torchvision
```
**For GPU (NVIDIA):** Install PyTorch with CUDA from https://pytorch.org/get-started/locally/

### Step 4: Change 3 settings in Cell 1, then run all cells top to bottom

---

## 📂 PROJECT FOLDER STRUCTURE (Created Automatically)
```
basketball_project/
├── datasets/          ← downloaded Roboflow training datasets
├── merged_dataset/    ← all datasets combined, cleaned, unified classes
├── runs/              ← training output (model weights, loss curves, charts)
├── extracted_frames/  ← frames sampled from test video
├── outputs/           ← tracked video, heatmaps, all analytics charts
└── weights/           ← copy of best trained model weights
```


---
## ⚙️ CELL 1 — Configuration & Environment Setup

### 🎯 PURPOSE
This cell does three things:
1. **Sets all your personal paths** (where to save files, your API key, your video)
2. **Creates the entire folder structure** automatically
3. **Detects your hardware** (GPU vs CPU) and sets appropriate settings

### ⚠️ YOU MUST CHANGE THESE 3 THINGS
| Setting | What to Change | Example |
|---|---|---|
| `BASE_DIR` | Where all project files are saved on YOUR machine | `'./basketball_project'` or `r'C:\Users\You\basketball'` |
| `API_KEY` | Your free Roboflow API key | `'abc123xyz...'` |
| `INPUT_VIDEO_PATH` | Path to a basketball video on your machine | `r'C:\Downloads\game.mp4'` or `''` to auto-download |

### 🔑 How to Get Your Roboflow API Key (Free)
1. Go to https://roboflow.com → Sign Up (free, no credit card)
2. Click profile icon (top right) → Settings
3. Click "Roboflow API" → Copy key → Paste below

### 📌 What the Code Does
- `os.makedirs(d, exist_ok=True)` — creates folders without error if they already exist
- `torch.cuda.is_available()` — returns True if NVIDIA GPU with CUDA is available
- `DEVICE = 0 if GPU else 'cpu'` — 0 means "first GPU", 'cpu' means use processor
- `BATCH_SIZE = 16 if GPU else 4` — larger batches fit in GPU memory, CPU needs smaller

### ✅ After Running This Cell
All empty project folders are created. Variables `BASE_DIR`, `MERGED_DIR`, `RUNS_DIR`, `OUTPUTS_DIR`, `BEST_MODEL_PATH`, `OUTPUT_VIDEO`, `INPUT_VIDEO` are set for all later cells.


In [ ]:
# ================================================================
# CELL 1 — LOCAL CONFIGURATION
# ================================================================
# ⚠️  CHANGE THESE 3 SETTINGS:
# ================================================================

# ══════════════════════════════════════════════════════════════
# 1. Where to save all project files
#    Windows: r'C:\Users\YourName\basketball_project'
#    Mac/Linux: '/home/yourname/basketball_project'
BASE_DIR = './basketball_project'

# 2. Your free Roboflow API key
#    Get: roboflow.com → Settings → Roboflow API
API_KEY = 'xkqvhgscUP22oMVy3tgu'

# 3. Path to your basketball input video
#    Windows: r'C:\Users\YourName\Downloads\game.mp4'
#    Mac:     '/Users/yourname/Downloads/game.mp4'
#    Leave '' to use auto-download in Cell 10
INPUT_VIDEO_PATH = r'C:\Users\MITHUN\Desktop\STUDIES\Drive\Big Vision\Code\Inputs\14800461_3840_2160_60fps.mp4'
# ══════════════════════════════════════════════════════════════

import os, cv2, shutil, glob, yaml, random, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from PIL import Image as PILImage
import torch, pandas as pd
warnings.filterwarnings('ignore')
sns.set_style('darkgrid')

# ── Create directory structure ────────────────────────────────
DATASET_DIR = os.path.join(BASE_DIR, 'datasets')
MERGED_DIR  = os.path.join(BASE_DIR, 'merged_dataset')
RUNS_DIR    = os.path.join(BASE_DIR, 'runs')
FRAMES_DIR  = os.path.join(BASE_DIR, 'extracted_frames')
OUTPUTS_DIR = os.path.join(BASE_DIR, 'outputs')
WEIGHTS_DIR = os.path.join(BASE_DIR, 'weights')

for d in [DATASET_DIR,MERGED_DIR,RUNS_DIR,FRAMES_DIR,OUTPUTS_DIR,WEIGHTS_DIR]:
    os.makedirs(d, exist_ok=True)

OUTPUT_VIDEO    = os.path.join(OUTPUTS_DIR, 'basketball_tracked.mp4')
BEST_MODEL_PATH = os.path.join(RUNS_DIR, 'yolov11m_basketball', 'weights', 'best.pt')
INPUT_VIDEO     = INPUT_VIDEO_PATH if INPUT_VIDEO_PATH and os.path.exists(INPUT_VIDEO_PATH) else None

# Device auto-detect
DEVICE     = 0 if torch.cuda.is_available() else 'cpu'
BATCH_SIZE = 16 if torch.cuda.is_available() else 4
WORKERS    = 4

print('='*55)
print('  LOCAL ENVIRONMENT CHECK')
print('='*55)
print(f'  BASE_DIR   : {os.path.abspath(BASE_DIR)}')
print(f'  PyTorch    : {torch.__version__}')
print(f'  GPU        : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  GPU Name   : {torch.cuda.get_device_name(0)}')
else:
    print('  CPU mode — training will be slow (~4-8hrs). GPU recommended.')
print(f'  INPUT VIDEO: {INPUT_VIDEO or "NOT SET — set in Cell 10"}')
print('='*55)


---
## 📥 CELL 2 — Download Training Datasets via Roboflow API

### 🎯 PURPOSE
Downloads 3 labeled basketball datasets directly from Roboflow to your local machine. These are the images the model will learn from.

### 📊 What Gets Downloaded
| Dataset | Images | Source | Why This Dataset |
|---|---|---|---|
| basketball-player-detection | ~1,616 | lokesh-podipireddy | Largest single source, diverse court angles |
| basketball-player-detection-2 | ~1,398 | roboflow-jvuqo | Different court types and lighting conditions |
| basketball-players-fy4c2 | ~600+ | Roboflow Official | High-quality Roboflow-curated annotations |

### 🏷️ What "Labeled" Means
Each image comes with a `.txt` label file. Each line in that file = one object:
```
0 0.512 0.438 0.089 0.201
↑ class_id   ↑ cx  ↑ cy  ↑ width  ↑ height
```
All values are normalized 0-1. `0` = player, `1` = ball.

### 📁 Where Files Go
`BASE_DIR/datasets/dataset1/`, `dataset2/`, `dataset3/`

### 🔑 Format: YOLOv8
We download in `yolov8` format. This is directly compatible with YOLOv11 (same format, different model version). Each dataset includes:
- `images/train/` — training images
- `labels/train/` — matching label files
- `images/valid/` — validation images
- `data.yaml` — class names and paths config

### ⚠️ Dependency
Your `API_KEY` from Cell 1 must be valid. If a dataset fails (workspace changed, etc.), the code skips it and uses whatever downloaded successfully.


In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 2 — DOWNLOAD ALL 3 DATASETS VIA ROBOFLOW API (FIXED)
# ════════════════════════════════════════════════════════════

import os
import glob
from roboflow import Roboflow

# ================= CONFIG =================
API_KEY = "xkqvhgscUP22oMVy3tgu"   # ← your API key
WORKSPACE_DIR = "./basketball_project/datasets"

os.makedirs(WORKSPACE_DIR, exist_ok=True)

rf = Roboflow(api_key=API_KEY)

DATASETS_CONFIG = [
    ("lokesh-podipireddy-eocdt", "basketball-player-detection-6y9yj", 1, "dataset1", "Dataset 1 (1616 imgs)"),
    ("roboflow-jvuqo", "basketball-player-detection-2", 1, "dataset2", "Dataset 2 (1398 imgs)"),
    ("roboflow-universe-projects", "basketball-players-fy4c2", 1, "dataset3", "Dataset 3 (Official)"),
]

DATASET_PATHS = []

print("\n🚀 STARTING DOWNLOADS...\n")

for ws, proj, ver, folder, desc in DATASETS_CONFIG:
    save_path = os.path.join(WORKSPACE_DIR, folder)

    print(f"📦 Downloading: {desc}")
    try:
        project = rf.workspace(ws).project(proj)
        version = project.version(ver)

        dataset = version.download("yolov8", location=save_path)

        # Count all image formats
        img_count = len(glob.glob(f"{save_path}/**/*.*", recursive=True))

        print(f"   ✅ Downloaded → {img_count} files at {save_path}")

        DATASET_PATHS.append(save_path)

    except Exception as e:
        print(f"   ❌ FAILED: {e}")

# ================= SUMMARY =================
total_images = sum(len(glob.glob(f"{p}/**/*.*", recursive=True)) for p in DATASET_PATHS)

print("\n═══════════════════════════════════════")
print(f"✅ Completed: {len(DATASET_PATHS)} datasets")
print(f"📊 Total files (approx): {total_images}")
print("═══════════════════════════════════════\n")

---
## 🔍 CELL 3 — Smart Path Detection + Dataset Inspection

### 🎯 PURPOSE
Solves the **"0 images found" bug** that breaks the merge step. Then inspects every dataset to confirm what downloaded.

### 🐛 The Bug This Fixes
Roboflow datasets use **two different folder layouts** depending on version:
```
Layout A (standard):  dataset/images/train/  +  dataset/labels/train/
Layout B (alternate): dataset/train/images/  +  dataset/train/labels/
```
Without this fix, the merge function checks only Layout A, finds 0 images in datasets using Layout B, and silently copies nothing.

### 🛠️ How `get_yolo_paths()` Works
```python
get_yolo_paths(root='/path/to/dataset1', split='train')
# Returns: ('/path/to/dataset1/images/train', '/path/to/dataset1/labels/train')
# OR:      ('/path/to/dataset1/train/images', '/path/to/dataset1/train/labels')
# OR:      (None, None) if neither layout has images
```
The function also handles `valid` vs `val` naming — Roboflow uses both interchangeably.

### 📊 Inspection Output
After running, you'll see for each dataset:
- Class names (what objects are labeled)
- Image count per split (train/valid/test)
- Exact folder path where images were found

### 🔑 Why This Cell Comes Before Merge
`get_yolo_paths()` is used in every subsequent cell (quality filter, preprocess, merge). It must be defined first.


In [ ]:
def get_yolo_paths(root, split):
    aliases = {'valid':['valid','val'],'val':['val','valid'],
               'train':['train'],'test':['test']}.get(split,[split])
    for s in aliases:
        img_a = os.path.join(root,'images',s)
        lbl_a = os.path.join(root,'labels',s)
        if os.path.exists(img_a) and (glob.glob(f'{img_a}/*.jpg')+glob.glob(f'{img_a}/*.png')):
            return img_a, lbl_a
        img_b = os.path.join(root,s,'images')
        lbl_b = os.path.join(root,s,'labels')
        if os.path.exists(img_b) and (glob.glob(f'{img_b}/*.jpg')+glob.glob(f'{img_b}/*.png')):
            return img_b, lbl_b
    return None, None
print('Smart path detection ready.')

for i,p in enumerate(DATASET_PATHS,1):
    print(f'\nDataset {i}: {p}')
    if not os.path.exists(p): print('  NOT FOUND'); continue
    yf=os.path.join(p,'data.yaml')
    if os.path.exists(yf):
        with open(yf) as f: info=yaml.safe_load(f)
        print(f'  Classes: {info.get("names",[])}')  
    t=0
    for sp in ['train','valid','test']:
        d,_=get_yolo_paths(p,sp)
        if d:
            n=len(glob.glob(f'{d}/*.jpg')+glob.glob(f'{d}/*.png'))
            print(f'  {sp}: {n}'); t+=n
    print(f'  TOTAL: {t}')

---
## 🧹 CELL 4 — Quality Filter (Remove Bad Training Images)

### 🎯 PURPOSE
Automatically scans every image in all 3 datasets and removes ones that would hurt model training. **Better to have 4,500 clean images than 5,000 noisy ones.**

### 🔎 What Gets Removed & Why
| Problem | Detection Method | Threshold | Why It Hurts Training |
|---|---|---|---|
| **Blurry frames** | Laplacian variance | score < 80 | Model learns blurry player shapes = misses sharp real players |
| **Too dark** | HSV Value channel mean | brightness < 25 | Model can't see features = learns noise |
| **Too bright/overexposed** | HSV Value channel mean | brightness > 235 | White-out frames have no detail |
| **Corrupted files** | PIL image verify() | any error | Crashes training if included |

### 🔬 How Blur Detection Works
```python
gray = cv2.imread(image, GRAYSCALE)
blur_score = cv2.Laplacian(gray, cv2.CV_64F).var()
# Sharp image:  score ~300-1000 (many strong edges)
# Blurry image: score < 80     (soft edges, low variance)
```
The Laplacian is a second-order derivative — it measures edge sharpness. High variance = many sharp edges = clear image.

### 🔬 How Brightness Detection Works
```python
hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
brightness = hsv[:, :, 2].mean()  # Channel 2 = Value = brightness
# Normal court image: brightness ~80-180
# Too dark: < 25   |  Too bright: > 235
```

### ⚖️ Paired Deletion (Critical!)
When an image is removed, its matching `.txt` label file is **always removed too**.
Having a label file with no matching image (orphan label) causes YOLOv11 to crash during training.

### 📊 Expected Result
Typically 2-8% of images are removed. More than 15% suggests a dataset quality issue.


In [ ]:
def quality_filter(dataset_path):
    removed, kept = 0, 0
    for split in ['train','valid','val','test']:
        img_dir, lbl_dir = get_yolo_paths(dataset_path, split)
        if not img_dir: continue
        images = glob.glob(f'{img_dir}/*.jpg')+glob.glob(f'{img_dir}/*.png')
        for ip in images:
            reason = None
            try: PILImage.open(ip).verify()
            except: reason = 'corrupted'
            if reason is None:
                gray = cv2.imread(ip, cv2.IMREAD_GRAYSCALE)
                if gray is None: reason = 'unreadable'
                else:
                    if cv2.Laplacian(gray,cv2.CV_64F).var() < 80: reason = 'blurry'
                    else:
                        bgr = cv2.imread(ip)
                        if bgr is not None:
                            b = cv2.cvtColor(bgr,cv2.COLOR_BGR2HSV)[:,:,2].mean()
                            if b < 25: reason='dark'
                            elif b > 235: reason='bright'
            if reason:
                os.remove(ip)
                if lbl_dir:
                    lp = os.path.join(lbl_dir, Path(ip).stem+'.txt')
                    if os.path.exists(lp): os.remove(lp)
                removed += 1
            else: kept += 1
    return kept, removed

print('Filtering...')
for i,p in enumerate(DATASET_PATHS,1):
    if os.path.exists(p):
        k,r=quality_filter(p); print(f'  Dataset {i}: kept={k}, removed={r}')
print('Done!')

---
## ⚙️ CELL 5 — Image Preprocessing (Resize to 640×640)

### 🎯 PURPOSE
Standardizes all images to exactly **640×640 pixels** and converts any grayscale images to RGB color. This ensures consistent input to YOLOv11m.

### 📐 Why 640×640?
| Reason | Explanation |
|---|---|
| YOLOv11 standard | The pretrained COCO weights were trained at 640×640. Using same size = maximum transfer learning benefit |
| Grid processing | YOLO divides image into NxN grids internally. Square input = equal grid cells in both axes |
| Memory efficiency | 640×640 × batch_16 fits exactly in T4/RTX GPU memory |
| Speed | 640 is power of 2 — GPU processes this optimally |

### ⚡ Why Labels Are NOT Changed
YOLO labels use **normalized 0–1 coordinates**, not pixel values.
```
# Before resize (1920×1080): label cx=0.5 = at pixel 960
# After resize  (640×640):   label cx=0.5 = at pixel 320
# The label file (0.5) stays identical — both mean "center of image"
```

### 🎨 Grayscale Conversion
Some datasets contain grayscale images (1 channel). YOLOv11m expects 3-channel BGR.
```python
if len(img.shape) == 2:               # 2D = grayscale
    img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)  # → 3-channel
```

### 💾 Interpolation Method
`cv2.INTER_LINEAR` is used for resizing — this is bilinear interpolation, the best balance between speed and quality for downscaling sports images.

### 📊 Expected Output
Most images will be "resized" (they come in various sizes from different cameras). A small number will be "converted" (grayscale). Zero errors = perfect.


In [ ]:
def preprocess_images(dataset_path):
    resized, converted, errors = 0, 0, 0
    for split in ['train','valid','val','test']:
        img_dir, _ = get_yolo_paths(dataset_path, split)
        if not img_dir: continue
        for ip in glob.glob(f'{img_dir}/*.jpg')+glob.glob(f'{img_dir}/*.png'):
            try:
                img = cv2.imread(ip)
                if img is None: errors+=1; continue
                if len(img.shape)==2:
                    img=cv2.cvtColor(img,cv2.COLOR_GRAY2BGR); converted+=1
                if (img.shape[1],img.shape[0])!=(640,640):
                    img=cv2.resize(img,(640,640),interpolation=cv2.INTER_LINEAR); resized+=1
                cv2.imwrite(ip,img,[cv2.IMWRITE_JPEG_QUALITY,95])
            except: errors+=1
    return resized, converted, errors

for i,p in enumerate(DATASET_PATHS,1):
    if os.path.exists(p):
        r,c,e=preprocess_images(p); print(f'Dataset {i}: resized={r}, converted={c}, errors={e}')
print('Done!')

---
## 🔀 CELL 6 — Merge All Datasets into One (High Accuracy Version)

### 🎯 PURPOSE
Combines all 3 datasets into a single `merged_dataset/` folder with:
- **Unified class IDs** (player=0, ball=1 — same across all datasets)
- **False detection filters** (removes post/billboard shaped boxes)
- **Conflict-free filenames** (prefixed with ds0_, ds1_, ds2_)
- **A single data.yaml config** that YOLOv11 reads for training

### 🏷️ Why Only 2 Classes? (Player + Ball)
**The referee class was removed deliberately:**
```
Problem found: Basketball posts and hoops were being detected as players.
Root cause:    Referees in dark uniforms near court equipment created a training
               pattern that generalized to vertical dark structures (posts, poles).
Fix:           Remove referee class → model learns cleaner player vs background boundary.
```

### 🔄 Class ID Remapping
Different datasets number classes differently. This function normalizes everything:
```python
def remap_cls(orig_id, src_map):
    name = src_map.get(orig_id, 'player').lower()
    if 'player' or 'person' or 'athlete' in name: return 0  # → player
    if 'ball' or 'basketball' in name:            return 1  # → ball
    return None   # referee, hoop, etc. → SKIP (not added to merged dataset)
```

### 📐 False Detection Filters (Aspect Ratio)
Applied to every bounding box during merge:
```
height / width > 5.5 → REJECT  (too tall/thin = basketball post shape)
width / height > 4.5 → REJECT  (too wide/flat = advertising board shape)
area < 0.5% of image → REJECT  (too small = distant noise)
Real player ratio:     height/width ≈ 1.5–4.0  (human body proportions)
```

### 📁 Filename Prefixing
```
dataset1/images/train/img001.jpg  → merged/images/train/ds0_img001.jpg
dataset2/images/train/img001.jpg  → merged/images/train/ds1_img001.jpg
```
Without prefixing, both files would overwrite each other as `img001.jpg`.

### 📄 data.yaml Created
```yaml
path: /path/to/merged_dataset
train: images/train
val: images/valid
test: images/test
nc: 2
names: [player, ball]
```
YOLOv11 reads this file before training to know: how many classes, what names, where images are.

### 📊 Expected Result
~3,000–4,000 total images across train/valid/test splits. The "skipped boxes" count shows how many false-detection boxes were removed from training data.


In [ ]:
TARGET_CLASSES = {0:'player', 1:'ball'}

def get_cls_map(yaml_path):
    if not os.path.exists(yaml_path): return {0:'player'}
    with open(yaml_path) as f: info = yaml.safe_load(f)
    return {i:n.lower() for i,n in enumerate(info.get('names',['player']))}

def remap_cls(orig_id, src_map):
    name = src_map.get(orig_id,'player').lower()
    if any(k in name for k in ['player','person','athlete']): return 0
    if any(k in name for k in ['ball','basketball']): return 1
    return None

def is_valid_box(cx, cy, bw, bh, cls_id):
    if bw*bh < 0.005: return False
    if cls_id==0:
        if bh>0 and bw/max(bh,1e-5) > 4.5: return False
        if bw>0 and bh/max(bw,1e-5) > 5.5: return False
    return True

def merge_datasets(src_dirs, target_dir, yaml_path_key='path'):
    os.makedirs(target_dir, exist_ok=True)
    for split in ['train','valid','test']:
        os.makedirs(f'{target_dir}/images/{split}', exist_ok=True)
        os.makedirs(f'{target_dir}/labels/{split}', exist_ok=True)
    counts={'train':0,'valid':0,'test':0}; skipped=0
    for di,src in enumerate(src_dirs):
        if not os.path.exists(src): continue
        src_map = get_cls_map(os.path.join(src,'data.yaml'))
        for tgt_split in ['train','valid','test']:
            img_dir,lbl_dir = get_yolo_paths(src,tgt_split)
            if not img_dir: continue
            images = glob.glob(f'{img_dir}/*.jpg')+glob.glob(f'{img_dir}/*.png')
            for ip in images:
                stem=Path(ip).stem; ext=Path(ip).suffix; nn=f'ds{di}_{stem}'
                shutil.copy(ip,f'{target_dir}/images/{tgt_split}/{nn}{ext}')
                sl=os.path.join(lbl_dir or '',stem+'.txt')
                dl=f'{target_dir}/labels/{tgt_split}/{nn}.txt'
                if os.path.exists(sl):
                    lines=[]
                    with open(sl) as f:
                        for line in f:
                            parts=line.strip().split()
                            if len(parts)!=5: continue
                            nc=remap_cls(int(parts[0]),src_map)
                            if nc is None: skipped+=1; continue
                            cx,cy,bw,bh=map(float,parts[1:])
                            if not is_valid_box(cx,cy,bw,bh,nc): skipped+=1; continue
                            lines.append(f'{nc} {chr(32).join(parts[1:])}\n')
                    with open(dl,'w') as f: f.writelines(lines)
                else: open(dl,'w').close()
                counts[tgt_split]+=1
    return counts, skipped

print('Merging...')
counts,skipped=merge_datasets(DATASET_PATHS,MERGED_DIR)
data_yaml={'path':MERGED_DIR,'train':'images/train','val':'images/valid','test':'images/test','nc':2,'names':['player','ball']}
with open(os.path.join(MERGED_DIR,'data.yaml'),'w') as f: yaml.dump(data_yaml,f,default_flow_style=False)
print(f'Skipped {skipped} false boxes')
for sp,n in counts.items(): print(f'  {sp}: {n}')
print(f'  TOTAL: {sum(counts.values())}')

---
## ✅ CELL 7 — Verify Annotations + Class Distribution Chart

### 🎯 PURPOSE
Two things in one cell:
1. **Validate** every single label file has correct YOLO format
2. **Visualize** class balance across train/valid/test splits as bar charts

### 🔍 Part 1: Annotation Verification
Every `.txt` label file is checked for:
| Check | Valid Example | Invalid Example |
|---|---|---|
| Exactly 5 values per line | `0 0.5 0.4 0.1 0.2` | `0 0.5 0.4 0.1` (only 4) |
| All coordinates 0.0–1.0 | `0 0.512 0.438 0.089 0.2` | `0 1.512 0.438...` (> 1.0) |
| Valid class IDs (0 or 1) | `0` or `1` | `3` or `-1` |
| Numeric values | `0.512` | `abc` |

**Why this matters:** A single malformed label file can silently corrupt training results or cause a crash mid-training (after 2 hours of compute).

### 📊 Part 2: Class Distribution Chart
Shows how many `player` and `ball` annotations exist in each split.

**What good class distribution looks like:**
- Player annotations: 5–20× more than ball (10 players per frame vs 1 ball)
- Consistent ratio across train/valid/test splits
- Total annotations: 10,000–30,000 is typical

**Why this matters:** If `ball` has 50× fewer annotations than `player`, the model will be bad at detecting the ball. You'd need to augment ball data or weight the loss function.

### 📁 Output
Chart saved to `OUTPUTS_DIR/class_distribution.png`


In [ ]:
errors,total,vc=[],0,0
for split in ['train','valid','test']:
    ldir=os.path.join(MERGED_DIR,'labels',split)
    if not os.path.exists(ldir): continue
    for lp in glob.glob(f'{ldir}/*.txt'):
        total+=1; ok=True
        with open(lp) as f:
            for ln,line in enumerate(f):
                parts=line.strip().split()
                if not parts: continue
                if len(parts)!=5: errors.append(f'{Path(lp).name}:{ln}'); ok=False
                else:
                    try:
                        cid=int(parts[0]); vals=list(map(float,parts[1:]))
                        if not all(0<=v<=1 for v in vals): errors.append(f'Range:{ln}'); ok=False
                    except: errors.append(f'Non-num:{ln}'); ok=False
        if ok: vc+=1
print(f'Labels: total={total}, valid={vc}, errors={len(errors)}')
counts={s:{0:0,1:0} for s in ['train','valid','test']}
for split in ['train','valid','test']:
    ldir=os.path.join(MERGED_DIR,'labels',split)
    if not os.path.exists(ldir): continue
    for lp in glob.glob(f'{ldir}/*.txt'):
        with open(lp) as f:
            for line in f:
                p=line.strip().split()
                if len(p)==5:
                    c=int(p[0])
                    if c in counts[split]: counts[split][c]+=1
fig,axes=plt.subplots(1,3,figsize=(15,5))
for ax,split in zip(axes,['train','valid','test']):
    vals=[counts[split][k] for k in [0,1]]
    bars=ax.bar(['player','ball'],vals,color=['#2196F3','#FF9800'],edgecolor='black',lw=0.5)
    for bar in bars:
        h=bar.get_height()
        if h>0: ax.text(bar.get_x()+bar.get_width()/2,h+5,f'{int(h):,}',ha='center',va='bottom',fontsize=10)
    ax.set_title(f'{split} — {sum(vals):,}')
plt.suptitle('Class Distribution',fontsize=13); plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_DIR,'class_distribution.png'),dpi=120,bbox_inches='tight'); plt.show()
print('Chart saved!')

---
## 🚀 CELL 8 — Train YOLOv11m (Fine-Tuning) ⭐ MAIN TRAINING STEP

### 🎯 PURPOSE
Fine-tunes YOLOv11m — a model pre-trained on Microsoft COCO — on your merged basketball dataset. This is where the model learns to recognize basketball players and the ball.

### 🧠 Fine-Tuning vs Training From Scratch
| Approach | Starting Point | Time | Data Needed | Result |
|---|---|---|---|---|
| **Fine-tuning ✅** | COCO weights (knows human shapes) | ~90 min | ~3,000 images | Excellent |
| From scratch | Random weights | Days–weeks | Millions of images | Needs massive data |

**COCO pretrained means:** The model already understands edges → shapes → human body proportions → person class. We just adapt it to basketball-specific context.

### ⚙️ Key Hyperparameters Explained
| Parameter | Value | Why This Value |
|---|---|---|
| `epochs=100` | 100 | Enough for convergence; patience=20 stops early if no improvement |
| `imgsz=640` | 640×640 | Matches COCO pretraining size → maximum transfer learning benefit |
| `batch=BATCH_SIZE` | 16 (GPU) / 4 (CPU) | Largest that fits in memory; bigger = more stable gradient updates |
| `optimizer='AdamW'` | AdamW | Better than SGD for fine-tuning pre-trained models (adaptive learning rate) |
| `lr0=0.001` | 0.001 | Low LR = gentle fine-tuning; preserves COCO knowledge instead of overwriting |
| `warmup_epochs=5` | 5 | Ramps LR from 0 to 0.001 over first 5 epochs — prevents initial instability |
| `patience=20` | 20 | Auto-stops if validation mAP doesn't improve for 20 consecutive epochs |
| `pretrained=True` | True | **Critical** — this IS fine-tuning; False = training from scratch |
| `flipud=0.0` | 0 | Never flip vertically — an upside-down basketball court is nonsense |
| `mosaic=1.0` | 1.0 | Combines 4 images into one training sample — excellent for crowded court scenes |

### 📈 Augmentations Applied During Training
| Augmentation | Basketball Relevance |
|---|---|
| HSV color jitter | Handles different court floors (maple, painted) and arena lighting |
| Horizontal flip (50%) | Left court vs right court attack direction symmetry |
| Scale ±50% | Players at different distances from camera |
| Mosaic | Creates artificially crowded scenes with 10+ players |
| Mixup | Blends two images — helps detect overlapping players |

### ⏱️ Expected Time
- **With GPU (NVIDIA RTX / T4):** ~60–90 minutes
- **Without GPU (CPU only):** 4–8 hours

### 📁 Output
Best model weights saved to: `RUNS_DIR/yolov11m_basketball/weights/best.pt`
A copy is also placed in `WEIGHTS_DIR/best_model.pt`

### ⚠️ If You Get "CUDA Out of Memory"
Change `batch=BATCH_SIZE` to `batch=8` or `batch=4` in the training call.


In [ ]:
from ultralytics import YOLO
model = YOLO('yolo11m.pt')
print('Training YOLOv11m...')
print('  Pre-trained on: COCO (Microsoft) — auto-downloads')
print('  Approach: Fine-tuning (NOT from scratch)')
results = model.train(
    data=os.path.join(MERGED_DIR,'data.yaml'),
    epochs=100, imgsz=640, batch=BATCH_SIZE, device=DEVICE, workers=WORKERS,
    optimizer='AdamW', lr0=0.001, lrf=0.01, momentum=0.937,
    weight_decay=0.0005, warmup_epochs=5,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, degrees=5.0,
    translate=0.1, scale=0.5, fliplr=0.5, flipud=0.0,
    mosaic=1.0, mixup=0.15, copy_paste=0.1,
    pretrained=True, patience=20, conf=0.001, iou=0.6,
    save=True, save_period=10, plots=True, val=True,
    project=RUNS_DIR, name='yolov11m_basketball', exist_ok=True, verbose=True)
BEST_MODEL_PATH=os.path.join(RUNS_DIR,'yolov11m_basketball','weights','best.pt')
shutil.copy(BEST_MODEL_PATH, os.path.join(WEIGHTS_DIR,'best_model.pt'))
print(f'\nTraining done! Best model: {BEST_MODEL_PATH}')


---
## 📊 CELL 9 — Evaluate Detection Accuracy + Advanced Analytics Graphs

### 🎯 PURPOSE
Measures how well the trained model detects players and ball on the **test set** (images the model has never seen). Generates 4 types of charts.

### 📏 Detection Metrics Explained
| Metric | Formula | What It Means | Target |
|---|---|---|---|
| **mAP@50** | Mean Average Precision at IoU=0.5 | Primary accuracy metric — at least 50% box overlap required for "correct" | > 0.85 |
| **mAP@50-95** | Mean AP across 10 IoU thresholds (0.5–0.95) | Stricter — requires tighter box fit | > 0.65 |
| **Precision** | TP / (TP + FP) | Of all detected players, what % are real? (false alarm rate) | > 0.85 |
| **Recall** | TP / (TP + FN) | Of all real players, what % did we find? (miss rate) | > 0.85 |
| **F1 Score** | 2×P×R / (P+R) | Harmonic mean — single balanced accuracy number | > 0.85 |

**IoU (Intersection over Union):** Measures box overlap.
`IoU = 0` = no overlap, `IoU = 1` = perfect overlap. mAP@50 means only boxes with ≥50% overlap count as correct detections.

### 📈 Charts Generated
| Chart | What It Shows | How to Read It |
|---|---|---|
| **Training Learning Curves** (6-panel) | Box loss, class loss, val loss, mAP@50, Precision, Recall over epochs | Loss should go DOWN. mAP/P/R should go UP. Flat lines = problem. |
| **Learning Rate Schedule** | How LR changes per epoch (warmup + cosine decay) | Rises for 5 epochs → smooth decay to near-zero |
| **Per-Class mAP Bar** | Separate mAP@50 for `player` and `ball` | Ball typically lower (it's smaller and faster) |
| **Overall Metrics Bar** | All 5 metrics in horizontal bar chart | All bars should cross the red 0.85 target line |

### 📌 Why `conf=0.001` During Evaluation?
During training `conf=0.001` is used (very low threshold). This includes ALL detections in the mAP calculation — even low-confidence ones — giving the most accurate mAP score. A high threshold would artificially inflate precision but miss low-confidence true positives.

### ⚠️ Dependency
Must run AFTER Cell 8 (training). The `BEST_MODEL_PATH` variable must point to an existing `.pt` file.


In [ ]:
from ultralytics import YOLO
best_model=YOLO(BEST_MODEL_PATH)
metrics=best_model.val(data=os.path.join(MERGED_DIR,'data.yaml'),
    split='test',conf=0.001,iou=0.6,plots=True,save_json=True,verbose=False)
p=metrics.box.mp; r=metrics.box.mr; f1=2*p*r/(p+r+1e-9)
print('='*55)
print('  DETECTION EVALUATION RESULTS')
print('='*55)
print(f'  mAP@50     : {metrics.box.map50:.4f}  (target > 0.85)')
print(f'  mAP@50-95  : {metrics.box.map:.4f}  (target > 0.65)')
print(f'  Precision  : {p:.4f}')
print(f'  Recall     : {r:.4f}')
print(f'  F1 Score   : {f1:.4f}')
print('-'*55)
per_class_map={}
for i,name in enumerate(['player','ball']):
    try:
        v=metrics.box.maps[i]; per_class_map[name]=v
        print(f'  {name:10s}: mAP50={v:.4f}  {chr(9608)*int(v*25)}')
    except: pass
print('='*55)
EVAL_RESULTS={'mAP50':metrics.box.map50,'mAP50_95':metrics.box.map,'precision':p,'recall':r,'f1':f1}
csv_path=os.path.join(RUNS_DIR,'yolov11m_basketball','results.csv')
if os.path.exists(csv_path):
    df_t=pd.read_csv(csv_path); df_t.columns=[c.strip() for c in df_t.columns]
    fig,axes=plt.subplots(2,3,figsize=(18,10))
    cfgs=[(axes[0,0],'train/box_loss','Box Loss Train','#e74c3c'),
          (axes[0,1],'train/cls_loss','Class Loss Train','#3498db'),
          (axes[0,2],'val/box_loss','Val Box Loss','#e67e22'),
          (axes[1,0],'metrics/mAP50(B)','mAP@50','#2ecc71'),
          (axes[1,1],'metrics/precision(B)','Precision','#9b59b6'),
          (axes[1,2],'metrics/recall(B)','Recall','#1abc9c')]
    for ax,col,title,color in cfgs:
        if col in df_t.columns:
            ax.plot(df_t.index,df_t[col],color=color,lw=2)
            ax.fill_between(df_t.index,df_t[col],alpha=0.15,color=color)
            ax.set_title(title,fontsize=12,fontweight='bold'); ax.set_xlabel('Epoch'); ax.grid(True,alpha=0.3)
    plt.suptitle('Training Learning Curves',fontsize=14); plt.tight_layout()
    plt.savefig(os.path.join(OUTPUTS_DIR,'learning_curves.png'),dpi=130,bbox_inches='tight'); plt.show()
    if 'lr/pg0' in df_t.columns:
        plt.figure(figsize=(10,4))
        plt.plot(df_t.index,df_t['lr/pg0'],color='#e74c3c',lw=2,label='LR Group 0')
        if 'lr/pg1' in df_t.columns: plt.plot(df_t.index,df_t['lr/pg1'],color='#3498db',lw=2,label='LR Group 1')
        plt.xlabel('Epoch'); plt.ylabel('Learning Rate'); plt.title('LR Schedule (Warmup+Cosine Decay)',fontsize=13)
        plt.legend(); plt.grid(True,alpha=0.3); plt.tight_layout()
        plt.savefig(os.path.join(OUTPUTS_DIR,'lr_schedule.png'),dpi=120); plt.show()
if per_class_map:
    fig,axes=plt.subplots(1,2,figsize=(14,5))
    bars=axes[0].bar(list(per_class_map.keys()),list(per_class_map.values()),color=['#2196F3','#FF9800'],edgecolor='black')
    for bar in bars:
        h=bar.get_height(); axes[0].text(bar.get_x()+bar.get_width()/2,h+0.01,f'{h:.3f}',ha='center',va='bottom',fontsize=12,fontweight='bold')
    axes[0].set_ylim(0,1.1); axes[0].axhline(y=0.85,color='red',ls='--',lw=1.5,label='Target 0.85')
    axes[0].set_title('Per-Class mAP@50',fontsize=13); axes[0].legend()
    mn=['mAP@50','mAP@50-95','Precision','Recall','F1']; mv=[metrics.box.map50,metrics.box.map,p,r,f1]
    cols=['#2196F3','#4CAF50','#9C27B0','#FF5722','#607D8B']
    bars2=axes[1].barh(mn,mv,color=cols,edgecolor='black')
    for bar in bars2:
        w=bar.get_width(); axes[1].text(w+0.005,bar.get_y()+bar.get_height()/2,f'{w:.4f}',va='center',fontsize=11,fontweight='bold')
    axes[1].set_xlim(0,1.15); axes[1].axvline(x=0.85,color='red',ls='--',lw=1.5,label='Target 0.85')
    axes[1].set_title('Overall Metrics',fontsize=13); axes[1].legend()
    plt.suptitle('Detection Accuracy Analysis',fontsize=14); plt.tight_layout()
    plt.savefig(os.path.join(OUTPUTS_DIR,'detection_metrics.png'),dpi=130,bbox_inches='tight'); plt.show()
print('Evaluation complete!')


---
## 🎬 CELL 10 — Set Input Video for Tracking

### 🎯 PURPOSE
Ensures the `INPUT_VIDEO` variable points to a valid basketball game video file on your local machine. The tracking pipeline (Cell 13) needs this.

### 📌 How Input Video Is Set (Priority Order)
1. **Cell 1 path** — if you set `INPUT_VIDEO_PATH` in Cell 1 and the file exists → used directly
2. **Manual override** — uncomment the `INPUT_VIDEO = r'...'` line in this cell
3. **Auto-download** — tries to download a free basketball video from Pexels

### 🎥 Getting a Basketball Video (3 Options)
| Option | How | Best When |
|---|---|---|
| **Use your own** | Set `INPUT_VIDEO_PATH` in Cell 1 | You have a local video file |
| **Manual Pexels** | Go to pexels.com/search/videos/basketball/ → Download any HD video | Auto-download fails |
| **Auto-download** | This cell tries 3 Pexels URLs automatically | Quick testing |

### ✅ What This Cell Prints
If video is found successfully:
```
Video ready: /path/to/basketball_game.mp4
  1920x1080 @ 30fps | 150.0s (4500 frames)
```
This confirms the video is readable and shows exactly what gets processed.

### ⚠️ If Auto-Download Fails
Networks in some environments block direct MP4 downloads. Options:
1. Download from https://www.pexels.com/search/videos/basketball/ manually
2. Uncomment and edit: `INPUT_VIDEO = r'C:\path\to\your\game.mp4'`


In [ ]:
if INPUT_VIDEO is None or not os.path.exists(str(INPUT_VIDEO)):
    print('INPUT_VIDEO not set. Options:')
    print('  1. Set INPUT_VIDEO_PATH in Cell 1 and re-run')
    print('  2. Uncomment and edit the line below:')
    # INPUT_VIDEO = r'C:\path\to\game.mp4'   # Windows
    # INPUT_VIDEO = '/path/to/game.mp4'          # Mac/Linux
    print('  3. Auto-downloading from Pexels...')
    import urllib.request
    test_vid = os.path.join(BASE_DIR,'basketball_game.mp4')
    try:
        url='https://videos.pexels.com/video-files/8425786/8425786-hd_1920_1080_25fps.mp4'
        req=urllib.request.Request(url,headers={'User-Agent':'Mozilla/5.0 Chrome/120.0'})
        with urllib.request.urlopen(req,timeout=60) as resp:
            with open(test_vid,'wb') as f: f.write(resp.read())
        mb=os.path.getsize(test_vid)/1e6
        if mb>1.0: INPUT_VIDEO=test_vid; print(f'Downloaded {mb:.1f} MB → {test_vid}')
    except Exception as e: print(f'Download failed: {e}')
if INPUT_VIDEO and os.path.exists(str(INPUT_VIDEO)):
    cap=cv2.VideoCapture(INPUT_VIDEO)
    fps=cap.get(cv2.CAP_PROP_FPS); w=int(cap.get(3)); h=int(cap.get(4))
    fr=int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); cap.release()
    print(f'Video ready: {INPUT_VIDEO}')
    print(f'  {w}x{h} @ {fps:.0f}fps | {fr/fps:.1f}s ({fr} frames)')

---
## 🎞️ CELL 11 — Extract Sample Frames from Video

### 🎯 PURPOSE
Samples individual frames from the input video at 5 FPS (1 frame every 6 frames at 30fps). Used for:
1. Visual inspection (did the video load correctly?)
2. Detection testing in Cell 12 (test model on still frames before full video processing)

### 📐 Sampling Rate: Why 5 FPS?
At 30fps video, consecutive frames are nearly identical. Taking 1 frame every 6 frames:
- Gives **diverse, non-redundant** frames for testing
- Covers the whole video without saving thousands of near-duplicate images
- Faster to process than extracting all 4,500 frames

### 🔎 Blur Filter Applied Here Too
```python
blur_score = cv2.Laplacian(gray, cv2.CV_64F).var()
if blur_score < 50: skip  # motion blur from fast player movement
```
Only sharp frames are saved for detection testing.

### 📁 Output
Frames saved to `FRAMES_DIR/frame_00001.jpg`, `frame_00002.jpg`, etc.
A 4-panel preview shows the first 4 extracted frames.

### 🔑 OpenCV Video Reading
```python
cap = cv2.VideoCapture(INPUT_VIDEO)
fps = cap.get(cv2.CAP_PROP_FPS)           # video framerate
total_frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)  # total frame count
ret, frame = cap.read()                   # read next frame
# ret = True if frame was read successfully, False at end of video
cap.release()                             # always release when done
```

### ⚠️ Dependency
Cell 10 must have set `INPUT_VIDEO` to a valid path.


In [ ]:
if INPUT_VIDEO and os.path.exists(str(INPUT_VIDEO)):
    os.makedirs(FRAMES_DIR,exist_ok=True)
    cap=cv2.VideoCapture(INPUT_VIDEO); vfps=cap.get(cv2.CAP_PROP_FPS)
    tf=int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); interval=max(1,int(vfps/5))
    saved=0; skipped=0
    for fi in range(tf):
        ret,frame=cap.read()
        if not ret: break
        if fi%interval!=0: continue
        gray=cv2.cvtColor(frame,cv2.COLOR_BGR2GRAY)
        if cv2.Laplacian(gray,cv2.CV_64F).var()<50: skipped+=1; continue
        cv2.imwrite(os.path.join(FRAMES_DIR,f'frame_{saved:05d}.jpg'),frame,[cv2.IMWRITE_JPEG_QUALITY,95])
        saved+=1
    cap.release(); print(f'Extracted {saved} frames (skipped {skipped} blurry)')
    
    # MODIFIED: Display 8 frames instead of 4, with larger font sizes
    flist=sorted(glob.glob(os.path.join(FRAMES_DIR,'*.jpg')))[:8]
    if flist:
        rows = (len(flist) + 3) // 4
        fig,axes=plt.subplots(rows,4,figsize=(20, 5 * rows))
        axes = axes.flatten() if rows > 1 else axes
        for idx, fp in enumerate(flist):
            img=cv2.imread(fp); axes[idx].imshow(cv2.cvtColor(img,cv2.COLOR_BGR2RGB))
            axes[idx].axis('off'); axes[idx].set_title(Path(fp).name,fontsize=12, fontweight='bold')
        # Hide unused subplots
        for idx in range(len(flist), len(axes)): axes[idx].axis('off')
        
        plt.suptitle('Sample Frames',fontsize=18, fontweight='bold'); plt.tight_layout()
        plt.savefig(os.path.join(OUTPUTS_DIR,'sample_frames.png'),dpi=100); plt.show()
else: print('Set INPUT_VIDEO in Cell 10 first.')

---
## 🔍 CELL 12 — Detection Test on Sample Frames

### 🎯 PURPOSE
Runs the trained YOLOv11m model on 6 still frames and draws bounding boxes. **Visual quality check before committing to full video processing** (~10–30 minutes).

### 📌 Why Test on Still Frames First?
If detection quality is poor on still frames, processing the full video will produce a bad output. Check here first — costs only seconds.

### ✅ What Good Detection Looks Like
- Players have tight bounding boxes (not too loose, not too small)
- Confidence scores > 0.50 for clearly visible players
- Ball detected when it's clearly in frame
- No boxes around court equipment (posts, hoops, scoreboards)

### ❌ What Bad Detection Looks Like and Why
| Problem | Likely Cause | Fix |
|---|---|---|
| No detections at all | Wrong model path, model not trained | Check BEST_MODEL_PATH, re-run Cell 8 |
| Court posts detected as players | Aspect ratio filter not working | Check Cell 6 merge ran correctly |
| Very low confidence (<0.3) | Not enough training, model underfitting | Increase epochs in Cell 8 |
| Boxes on advertising boards | Need more training data showing court perimeter | Add more diverse datasets |

### 🎨 Color Coding
- 🟠 **Orange** = Player (BGR: 255,120,0)
- 🩵 **Cyan** = Ball (BGR: 0,200,255)

### 🛡️ Fallback Model
```python
m_path = BEST_MODEL_PATH if os.path.exists(BEST_MODEL_PATH) else 'yolo11m.pt'
```
If training hasn't run yet, uses base COCO model as fallback so the cell still works.

### 📁 Output
Chart saved to `OUTPUTS_DIR/detection_test.png`


In [ ]:
from ultralytics import YOLO
import os
import glob
import cv2
import matplotlib.pyplot as plt

# Fallback to base model if best.pt is not found
m_path = BEST_MODEL_PATH if os.path.exists(BEST_MODEL_PATH) else 'yolo11m.pt'
best_model = YOLO(m_path)
sframes = sorted(glob.glob(os.path.join(FRAMES_DIR, '*.jpg')))[:6]

if not sframes: 
    print('Run Cell 11 first.')
else:
    # High-contrast colors: Player (Neon Green), Ball (Bright Red)
    # BGR format: Green (0, 255, 0), Red (0, 0, 255)
    CLS_C = {0: (0, 255, 0), 1: (0, 0, 255)} 
    CLS_N = {0: 'PLAYER', 1: 'BALL'}
    
    fig, axes = plt.subplots(2, 3, figsize=(24, 14)) # Increased figure size
    axes = axes.flatten()
    
    for idx, fp in enumerate(sframes):
        frame = cv2.imread(fp)
        result = best_model.predict(frame, conf=0.40, iou=0.5, verbose=False)[0]
        
        for box in result.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            cid = int(box.cls[0])
            conf = float(box.conf[0])
            
            # Select color and build label
            color = CLS_C.get(cid, (255, 255, 255))
            label = f'{CLS_N.get(cid,"?")} {conf:.2f}'
            
            # --- MAXIMUM VISIBILITY SETTINGS ---
            # 1. Ultra-Thick Bounding Box (Thickness 8)
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 8)
            
            # 2. Large Font Scale (1.2) and Thick Text (3)
            f_scale = 1.2
            f_thick = 3
            (tw, th), baseline = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, f_scale, f_thick)
            
            # 3. Solid Black background for the label to make text pop
            # Positioned slightly above the box
            cv2.rectangle(frame, (x1, y1 - th - 20), (x1 + tw + 10, y1), (0, 0, 0), -1)
            
            # 4. White text on the Black background
            cv2.putText(frame, label, (x1 + 5, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, f_scale, (255, 255, 255), f_thick)
        
        axes[idx].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        axes[idx].axis('off')
        axes[idx].set_title(f'Frame {idx+1} — {len(result.boxes)} Detections', fontsize=14, fontweight='bold')
        
    for idx in range(len(sframes), 6): 
        axes[idx].axis('off')
        
    plt.suptitle('YOLOv11m - HIGH VISIBILITY DETECTION TEST', fontsize=20, fontweight='bold', y=0.95)
    plt.tight_layout()
    # Save with high DPI for clarity
    plt.savefig(os.path.join(OUTPUTS_DIR, 'detection_test_ultra_bold.png'), dpi=200, bbox_inches='tight')
    plt.show()


---
## 🏃 CELL 13 — Full Player Tracking with ByteTrack ⭐ MAIN TRACKING STEP

### 🎯 PURPOSE
Processes every single frame of the input video:
1. **YOLOv11m detects** players and ball in each frame
2. **ByteTrack links** detections across frames (assigns consistent IDs)
3. **Supervision draws** bounding boxes, ID labels, and 50-frame movement trails
4. **Saves** the fully annotated output video

### 🧠 ByteTrack — How It Actually Works Here
```python
result = best_model.track(
    frame,
    tracker='bytetrack.yaml',  # ← activates ByteTrack
    persist=True,              # ← keeps track state between frames
    conf=0.40,                 # ← high-conf threshold for Stage 1
    iou=0.5,                   # ← NMS threshold
)
```
**What happens inside per frame:**
1. YOLOv11m runs on `frame` → returns boxes with confidence scores
2. Boxes split: high-conf (≥0.40) and low-conf (0.10–0.40)
3. Stage 1: Hungarian algorithm matches high-conf boxes to existing tracks by IoU
4. Stage 2: Remaining unmatched tracks are matched to low-conf boxes
5. Kalman filter predicts next-frame positions for all tracks
6. New tracks started for unmatched high-conf boxes (new players entering frame)

**Why `persist=True` is critical:**
Without it, ByteTrack creates a new instance every frame → IDs reset every frame → no tracking.
With it, ByteTrack remembers all tracks across the entire video.

### 📐 Post-Processing Filter (Happens After ByteTrack)
Even after training, the model can occasionally detect a post. This filter catches remaining ones:
```python
for i in range(len(dets)):
    x1, y1, x2, y2 = dets.xyxy[i]
    bw = x2 - x1;  bh = y2 - y1
    if class_id == 0 (player):
        if bh/bw > 5.5: skip  # too tall/thin = post
        if bw/bh > 4.5: skip  # too wide = billboard
```

### 🎨 What Appears in the Output Video
| Visual Element | What It Shows |
|---|---|
| Colored bounding box | Detection + class (Blue=Player, Cyan=Ball) |
| `#ID ClassName Confidence` label | e.g., `#3 Player 0.87` |
| Colored trail line | Last 50 positions of that player's movement path |
| Black HUD bar at top | Frame number, player count, algorithm name |

### 📊 Tracking Data Collected
Every detection is stored: `{frame, track_id, class_id, confidence, bounding_box}`. Used in Cells 14 and 15 for analytics.

### ⏱️ Expected Time
- 30fps × 150 second video = 4,500 frames
- GPU: ~5–8 minutes | CPU: ~20–40 minutes

### 📁 Output
Tracked video saved to `OUTPUTS_DIR/basketball_tracked.mp4`


In [ ]:
import supervision as sv
from ultralytics import YOLO
import os
import cv2
import numpy as np

best_model  = YOLO(BEST_MODEL_PATH)
CLASS_NAMES = {0:'Player', 1:'Ball'}

if INPUT_VIDEO is None or not os.path.exists(str(INPUT_VIDEO)):
    print(f'Video not found: {INPUT_VIDEO}')
else:
    if not os.path.exists(BEST_MODEL_PATH):
        print(f'ERROR: Best model not found at {BEST_MODEL_PATH}. Run Cell 8 first or check your path.')
    else:
        video_info = sv.VideoInfo.from_video_path(INPUT_VIDEO)
        print(f'Processing: {video_info.width}x{video_info.height} @ {video_info.fps}fps | {video_info.total_frames} frames')

        # --- BOLDER VISUALS ---
        box_ann   = sv.BoxAnnotator(thickness=6) 
        lbl_ann   = sv.LabelAnnotator(text_scale=1.0, text_thickness=2, text_padding=10)
        trace_ann = sv.TraceAnnotator(thickness=6, trace_length=50)
        tracking_data = []

        with sv.VideoSink(OUTPUT_VIDEO, video_info) as sink:
            for fi, frame in enumerate(sv.get_video_frames_generator(INPUT_VIDEO)):
                # ByteTrack detection + tracking
                result = best_model.track(
                    frame, tracker='bytetrack.yaml', 
                    persist=True, conf=0.40, iou=0.5, verbose=False)[0]
                
                dets = sv.Detections.from_ultralytics(result)
                
                # Post-filter: remove post/hoop shaped boxes
                if len(dets) > 0:
                    keep = []
                    for i in range(len(dets)):
                        x1, y1, x2, y2 = dets.xyxy[i]
                        bw = x2 - x1
                        bh = y2 - y1
                        if int(dets.class_id[i]) == 0:
                            if bh > 0 and bh / max(bw, 1) > 5.5: continue
                            if bw > 0 and bw / max(bh, 1) > 4.5: continue
                        keep.append(i)
                    if keep: 
                        dets = dets[keep]
                
                labels = []
                for i in range(len(dets)):
                    tid = int(dets.tracker_id[i]) if dets.tracker_id is not None else -1
                    cls = int(dets.class_id[i])
                    conf = float(dets.confidence[i])
                    labels.append(f'#{tid} {CLASS_NAMES.get(cls,"?")} {conf:.2f}')
                    if dets.tracker_id is not None:
                        tracking_data.append({
                            'frame': fi, 'track_id': tid,
                            'class_id': cls, 'conf': conf, 'xyxy': dets.xyxy[i].tolist()
                        })
                
                ann = frame.copy()
                
                # --- BUG FIX: Only annotate trace if tracker_id exists ---
                if dets.tracker_id is not None:
                    ann = trace_ann.annotate(scene=ann, detections=dets)
                
                ann = box_ann.annotate(scene=ann, detections=dets)
                ann = lbl_ann.annotate(scene=ann, detections=dets, labels=labels)
                
                # Header Overlay
                np_ = sum(1 for c in dets.class_id if c == 0)
                cv2.rectangle(ann, (0, 0), (650, 45), (0, 0, 0), -1)
                cv2.putText(ann, f'Frame:{fi:4d} | Players:{np_} | YOLOv11m + ByteTrack',
                            (10, 32), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2)
                
                sink.write_frame(ann)
                
                if fi % 60 == 0:
                    pct = fi / video_info.total_frames * 100
                    print(f'  {fi}/{video_info.total_frames} ({pct:.0f}%) det={len(dets)}')

        print(f'\nTracking complete! Output: {OUTPUT_VIDEO}')
        print(f'Total tracking entries: {len(tracking_data)}')


---
## 📊 CELL 14 — Tracking Analytics Dashboard (6 Charts)

### 🎯 PURPOSE
Analyzes the tracking data from Cell 13 and generates a 6-panel analytics dashboard showing tracking quality metrics.

### 📈 The 6 Charts Explained
| Chart | X-axis | Y-axis | What to Look For |
|---|---|---|---|
| **Track Length Distribution** | Track length (frames) | Number of tracks | Mostly long bars (right side) = good tracking. Many short bars = ID switches. |
| **Confidence by Class** | Confidence score (0–1) | Count | Player peak at 0.7–0.9 = strong detections. Ball peak may be lower (small object). |
| **Detections Per Frame** | Frame number | Total detections | Smooth curve. Spikes = crowded plays. Dips = players off-camera. |
| **Players Per Frame** | Frame number | Player count | Should average near 10 (full team visible). Red dashed line = mean. |
| **Track Length CDF** | Track length (frames) | Cumulative % | Gradual rise = many long tracks = good ByteTrack performance. |
| **Rolling Avg Confidence** | Frame number | 30-frame avg confidence | Should stay above 0.5. Dips = heavy occlusion periods. |

### 📊 Tracking Quality Metrics Printed
| Metric | What It Means | Target |
|---|---|---|
| **Total detections** | Every detection across all frames | Depends on video length |
| **Unique track IDs** | How many distinct players were identified | Should be 10–15 for a game clip |
| **Avg track length** | How many frames each player was continuously tracked | Higher = better |
| **Short tracks (<5f)** | Tracks under 5 frames = likely ID switches | As few as possible |
| **Long tracks (≥20f)** | Tracks over 20 frames = stable tracking | Should be 60%+ |
| **Approx MOTA** | Quick MOTA estimate without ground truth | Closer to 1.0 = better |

### 📁 Output
Chart saved to `OUTPUTS_DIR/tracking_analytics.png`

### ⚠️ Dependency
Cell 13 (tracking) must have run and populated `tracking_data` list.


In [ ]:
if 'tracking_data' not in dir() or not tracking_data:
    print('Run tracking cell first.')
else:
    df=pd.DataFrame(tracking_data)
    player_df=df[df['class_id']==0]
    track_lens=df.groupby('track_id').size()
    short=(track_lens<5).sum()
    long_pct=(track_lens>=20).sum()/max(len(track_lens),1)*100
    mota_approx=max(0.0,1.0-(short/max(len(df),1)))

    print('='*55)
    print('  TRACKING STATISTICS (ByteTrack)')
    print('='*55)
    print(f'  Detections  : {len(df):,}')
    print(f'  Track IDs   : {df["track_id"].nunique()}')
    print(f'  Avg conf    : {df["conf"].mean():.4f}')
    print(f'  Avg track   : {track_lens.mean():.1f} frames')
    print(f'  Short (<5f) : {short}')
    print(f'  Long (>=20f): {long_pct:.1f}%')
    print(f'  Approx MOTA : {mota_approx:.4f}')
    print('='*55)

    fig, axes = plt.subplots(2,3,figsize=(18,11))
    axes[0,0].hist(track_lens.values,bins=30,color='#2196F3',edgecolor='black',lw=0.5)
    axes[0,0].axvline(x=5,color='red',ls='--',lw=2,label='<5=ID switch')
    axes[0,0].axvline(x=20,color='green',ls='--',lw=2,label='>=20=stable')
    axes[0,0].set_title('Track Length Distribution',fontsize=11,fontweight='bold')
    axes[0,0].legend()
    for cid,nm,col in [(0,'Player','#2196F3'),(1,'Ball','#FF9800')]:
        sub=df[df['class_id']==cid]
        if len(sub)>0:
            axes[0,1].hist(sub['conf'].values,bins=25,alpha=0.65,label=nm,color=col,edgecolor='black',lw=0.3)
    axes[0,1].set_title('Confidence by Class',fontsize=11,fontweight='bold'); axes[0,1].legend()
    dpf=df.groupby('frame').size()
    axes[0,2].plot(dpf.index,dpf.values,color='#9C27B0',lw=1.2)
    axes[0,2].fill_between(dpf.index,dpf.values,alpha=0.2,color='#9C27B0')
    axes[0,2].set_title('Detections Per Frame',fontsize=11,fontweight='bold')
    ppf=player_df.groupby('frame').size()
    axes[1,0].plot(ppf.index,ppf.values,color='#2196F3',lw=1.5)
    axes[1,0].axhline(y=ppf.mean(),color='red',ls='--',lw=1.5,label=f'Mean={ppf.mean():.1f}')
    axes[1,0].set_title('Players Per Frame',fontsize=11,fontweight='bold'); axes[1,0].legend()
    sl=np.sort(track_lens.values); cdf=np.arange(1,len(sl)+1)/len(sl)
    axes[1,1].plot(sl,cdf,color='#4CAF50',lw=2); axes[1,1].axvline(x=20,color='red',ls='--',lw=1.5)
    axes[1,1].set_title('Track Length CDF',fontsize=11,fontweight='bold')
    rc=df.groupby('frame')['conf'].mean().rolling(30).mean()
    axes[1,2].plot(rc.index,rc.values,color='#FF5722',lw=1.5)
    axes[1,2].set_title('Rolling Avg Confidence (30f)',fontsize=11,fontweight='bold'); axes[1,2].set_ylim(0,1)
    plt.suptitle('ByteTrack Analytics Dashboard',fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUTS_DIR,'tracking_analytics.png'),dpi=130,bbox_inches='tight')
    plt.show()
    print('Tracking analytics saved!')


---
## 🔥 CELL 15 — Spatial Analytics Dashboard (Heatmaps + Zone Analysis)

### 🎯 PURPOSE
Generates a 6-panel spatial dashboard showing **WHERE things happened** on the court. Provides real-world basketball tactical insights from the tracked data.

### 🗺️ The 6 Spatial Charts Explained
| Chart | What It Shows | How It's Built | Tactical Insight |
|---|---|---|---|
| **Player Heatmap** | Where players spent most time on court | 2D position histogram → Gaussian blur → JET colormap overlaid on first frame | Red zones = high player traffic (paint, wing positions) |
| **Ball Heatmap** | Where the ball traveled | Same as player heatmap, ball class only, HOT colormap | Shows ball movement patterns, pick-and-roll zones |
| **Zone Occupancy Pie** | % of player time in 6 court zones | Court divided into Left/Right × Paint/Mid/3pt | Reveals team strategy — heavy paint = post-up, heavy 3pt = perimeter |
| **Player Density Grid** | 10×6 grid showing player count per cell | Normalize positions to grid, count accumulate, seaborn heatmap | High-resolution positioning analysis |
| **Player Trajectories** | Individual paths of first 5 players | Plot (cx,cy) sequence per track_id, different color per player | Shows individual player movement, screens set, cuts made |
| **Active Players Timeline** | How many players visible each frame | Count player detections per frame, area fill chart | Consistent ~10 = full-game clip. Dips = camera cuts or full-court plays |

### 🔬 How Heatmaps Are Built
```python
heatmap = np.zeros((H, W), dtype=np.float32)
for each player detection:
    cx = (x1 + x2) / 2    # center x
    cy = (y1 + y2) / 2    # center y
    heatmap[cy, cx] += 1  # increment at player position

heatmap = cv2.GaussianBlur(heatmap, (61, 61), 0)  # smooth it out
heatmap = normalize to 0-255
colored = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
overlay = cv2.addWeighted(first_frame, 0.35, colored, 0.65, 0)
```

### 📁 Output Files
- `OUTPUTS_DIR/player_heatmap.jpg` — player heatmap overlay
- `OUTPUTS_DIR/spatial_analytics.png` — full 6-panel dashboard

### ⚠️ Dependency
Cell 13 (tracking) must have run. Also needs `INPUT_VIDEO` for reading court background.


In [ ]:
if 'tracking_data' not in dir() or not tracking_data:
    print('Run tracking cell first.')
else:
    cap=cv2.VideoCapture(INPUT_VIDEO)
    W=int(cap.get(3)); H=int(cap.get(4)); ret,bg=cap.read(); cap.release()
    df=pd.DataFrame(tracking_data)
    player_df=df[df['class_id']==0]; ball_df=df[df['class_id']==1]

    def build_hm(df_sub):
        hm=np.zeros((H,W),dtype=np.float32)
        for _,row in df_sub.iterrows():
            x1,y1,x2,y2=row['xyxy']; cx=int((x1+x2)/2); cy=int((y1+y2)/2)
            if 0<=cy<H and 0<=cx<W: hm[cy,cx]+=1
        hm=cv2.GaussianBlur(hm,(61,61),0)
        if hm.max()>0: hm=(hm/hm.max()*255).astype(np.uint8)
        return hm

    p_hm=build_hm(player_df); b_hm=build_hm(ball_df) if len(ball_df)>0 else np.zeros((H,W),np.uint8)
    fig=plt.figure(figsize=(20,14))
    ax1=fig.add_subplot(2,3,1)
    cp=cv2.applyColorMap(p_hm,cv2.COLORMAP_JET)
    ov=cv2.addWeighted(bg,0.35,cp,0.65,0) if bg is not None else cp
    cv2.imwrite(os.path.join(OUTPUTS_DIR,'player_heatmap.jpg'),ov)
    ax1.imshow(cv2.cvtColor(ov,cv2.COLOR_BGR2RGB))
    ax1.set_title('Player Movement Heatmap',fontsize=12,fontweight='bold'); ax1.axis('off')
    ax2=fig.add_subplot(2,3,2)
    cb=cv2.applyColorMap(b_hm,cv2.COLORMAP_HOT)
    ax2.imshow(cv2.cvtColor(cb,cv2.COLOR_BGR2RGB))
    ax2.set_title('Ball Movement Heatmap',fontsize=12,fontweight='bold'); ax2.axis('off')
    ax3=fig.add_subplot(2,3,3)
    zones={'Left Paint':0,'Left Mid':0,'Left 3pt':0,'Right Paint':0,'Right Mid':0,'Right 3pt':0}
    for _,row in player_df.iterrows():
        x1,y1,x2,y2=row['xyxy']; cx=(x1+x2)/2/W; cy=(y1+y2)/2/H
        side='Left' if cx<0.5 else 'Right'
        if cy<0.35 or cy>0.65: zones[f'{side} Paint']+=1
        elif 0.35<=cy<=0.65 and ((side=='Left' and cx<0.33) or (side=='Right' and cx>0.67)): zones[f'{side} Mid']+=1
        else: zones[f'{side} 3pt']+=1
    zc=['#FF6B6B','#FFA07A','#FFD700','#90EE90','#87CEEB','#DDA0DD']
    ax3.pie(zones.values(),labels=zones.keys(),colors=zc,autopct='%1.1f%%',startangle=90)
    ax3.set_title('Zone Occupancy',fontsize=12,fontweight='bold')
    ax4=fig.add_subplot(2,3,4)
    gw,gh=10,6; density=np.zeros((gh,gw))
    for _,row in player_df.iterrows():
        x1,y1,x2,y2=row['xyxy']
        cx=min(int((x1+x2)/2/W*(gw-1)),gw-1); cy=min(int((y1+y2)/2/H*(gh-1)),gh-1)
        density[cy,cx]+=1
    sns.heatmap(density,ax=ax4,cmap='YlOrRd',annot=True,fmt='.0f',cbar=True)
    ax4.set_title('Player Density Grid',fontsize=12,fontweight='bold')
    ax5=fig.add_subplot(2,3,5)
    if bg is not None: ax5.imshow(cv2.cvtColor(bg,cv2.COLOR_BGR2RGB),alpha=0.4)
    tids=player_df['track_id'].unique()[:5]; tc=plt.cm.Set1(np.linspace(0,1,max(len(tids),1)))
    for tid,color in zip(tids,tc):
        traj=player_df[player_df['track_id']==tid].sort_values('frame')
        xs=[(r[0]+r[2])/2 for r in traj['xyxy']]; ys=[(r[1]+r[3])/2 for r in traj['xyxy']]
        ax5.plot(xs,ys,color=color,lw=2,label=f'Player #{tid}',alpha=0.8)
        if xs: ax5.scatter(xs[0],ys[0],color=color,s=80,zorder=5)
    ax5.set_xlim(0,W); ax5.set_ylim(H,0)
    ax5.set_title('Player Trajectories (first 5)',fontsize=12,fontweight='bold'); ax5.legend(fontsize=8); ax5.axis('off')
    ax6=fig.add_subplot(2,3,6)
    fr_range=range(int(df['frame'].max())+1); active=[len(player_df[player_df['frame']==f]) for f in fr_range]
    ax6.fill_between(fr_range,active,alpha=0.5,color='#2196F3')
    ax6.plot(fr_range,active,color='#1565C0',lw=1.5)
    ax6.set_title('Active Players Over Time',fontsize=12,fontweight='bold')
    plt.suptitle('Spatial Analytics Dashboard',fontsize=15)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUTS_DIR,'spatial_analytics.png'),dpi=130,bbox_inches='tight')
    plt.show()
    print('Spatial analytics saved!')


---
## 💾 CELL 16 — Save & Verify All Outputs

### 🎯 PURPOSE
Lists every output file the project has generated with a ✅/⚠️ status indicator. Since all files are saved to `OUTPUTS_DIR` as they're created, this cell just verifies everything is present.

### 📁 What Files Are Checked
| File | Created By Cell | What It Contains |
|---|---|---|
| `basketball_tracked.mp4` | Cell 13 | Output video with player IDs, boxes, movement trails |
| `best_model_yolov11m.pt` | Cell 8 | Trained model weights — the main deliverable |
| `best_model.pt` (copy) | Cell 8 | Duplicate in WEIGHTS_DIR for easy access |
| `player_heatmap.jpg` | Cell 15 | Court heatmap showing player positions |
| `spatial_analytics.png` | Cell 15 | 6-panel spatial dashboard |
| `tracking_analytics.png` | Cell 14 | 6-panel tracking quality dashboard |
| `learning_curves.png` | Cell 9 | Training loss + mAP curves over epochs |
| `detection_metrics.png` | Cell 9 | Accuracy bar charts (mAP, P, R, F1) |
| `lr_schedule.png` | Cell 9 | Learning rate schedule visualization |
| `class_distribution.png` | Cell 7 | Dataset class balance bar charts |

### 📌 Local vs Colab Storage
Unlike the Colab version (which needs to save to Drive before session ends), local files **persist permanently** on your machine. The project folder stays intact between sessions.

### 📤 Sharing Your Results
To share with evaluators:
1. Zip the entire `BASE_DIR/` folder
2. Share `OUTPUTS_DIR/basketball_tracked.mp4` as video demo
3. Share `WEIGHTS_DIR/best_model.pt` as model file
4. Share `RUNS_DIR/yolov11m_basketball/` for training evidence


In [ ]:
print(f'Project directory: {os.path.abspath(BASE_DIR)}')
print()
for path, label in [
    (OUTPUT_VIDEO,                                              'Tracked video'),
    (BEST_MODEL_PATH,                                           'Best model weights'),
    (os.path.join(WEIGHTS_DIR,'best_model.pt'),                 'Model copy'),
    (os.path.join(OUTPUTS_DIR,'player_heatmap.jpg'),            'Player heatmap'),
    (os.path.join(OUTPUTS_DIR,'spatial_analytics.png'),         'Spatial analytics'),
    (os.path.join(OUTPUTS_DIR,'tracking_analytics.png'),        'Tracking analytics'),
    (os.path.join(OUTPUTS_DIR,'learning_curves.png'),           'Learning curves'),
    (os.path.join(OUTPUTS_DIR,'detection_metrics.png'),         'Detection metrics'),
    (os.path.join(OUTPUTS_DIR,'lr_schedule.png'),               'LR schedule'),
    (os.path.join(OUTPUTS_DIR,'class_distribution.png'),        'Class distribution'),
]:
    status='✅' if os.path.exists(str(path)) else '⚠️ not found'
    print(f'  {status}  {label}: {path}')
if 'EVAL_RESULTS' in dir():
    print()
    print('Detection Results:')
    for k,v in EVAL_RESULTS.items(): print(f'  {k}: {v:.4f}')

---
## 🎨 CELL 17 — Gradio Live Demo UI (Local Web Interface)

### 🎯 PURPOSE
Launches an interactive web interface at **http://localhost:7860** where anyone can:
- Upload any basketball video (drag-and-drop)
- Click "Track Video"
- Watch the tracked output video appear in the browser

**No coding knowledge needed** to use the demo once this cell is running.

### 🌐 How Gradio Works Locally
```python
demo.launch(share=False, inbrowser=True)
# share=False  → only accessible on YOUR machine at localhost:7860
# inbrowser=True → opens browser tab automatically
```
Unlike the Colab version (`share=True` creates public HTTPS URL), local version runs only on your machine for privacy.

### 🔄 What Happens When You Upload a Video
1. Gradio saves uploaded file to a temp directory
2. `track_video()` function runs the full ByteTrack pipeline on it
3. Annotated output video is written to another temp file
4. Gradio displays the output video in the browser

### 🛠️ Technical Notes
- `asyncio` noise suppression: Windows asyncio sometimes prints `ConnectionResetError` when Gradio shuts down. The `_quiet` handler silently ignores these.
- `demo.close()` at start: prevents "port already in use" error if cell is re-run
- Output videos are saved to `OUTPUTS_DIR/` with a counter suffix (`gradio_output_1.mp4`, etc.)

### 🔑 ByteTrack Same Pipeline
The tracking inside the Gradio function uses the **identical code** as Cell 13:
```python
result = demo_model.track(frame, tracker='bytetrack.yaml', persist=True, conf=0.40)
```
Same model, same algorithm, same aspect ratio filter — results are identical.

### ⚠️ Keep This Cell Running
The Gradio server runs in the background. Keep Jupyter/the terminal running while using the demo. Stop it by restarting the kernel or running `demo.close()`.


In [ ]:
import gradio as gr
from ultralytics import YOLO
import supervision as sv
import os, cv2, numpy as np, shutil, asyncio, subprocess

# ── Suppress Windows asyncio noise ──
def _quiet(loop, ctx):
    if isinstance(ctx.get("exception"), (ConnectionResetError, PermissionError)):
        return
    loop.default_exception_handler(ctx)
try:
    asyncio.get_event_loop().set_exception_handler(_quiet)
except:
    pass

try:
    demo.close()
except:
    pass

ABS_OUTPUTS = os.path.abspath(OUTPUTS_DIR)
os.makedirs(ABS_OUTPUTS, exist_ok=True)

m_path = BEST_MODEL_PATH if os.path.exists(BEST_MODEL_PATH) else 'yolo11m.pt'
demo_model = YOLO(m_path)

# ── Find ffmpeg (bundled with imageio or on system PATH) ──
def find_ffmpeg():
    try:
        import imageio_ffmpeg
        return imageio_ffmpeg.get_ffmpeg_exe()
    except:
        pass
    for cmd in ["ffmpeg", "ffmpeg.exe"]:
        try:
            subprocess.run([cmd, "-version"], capture_output=True, timeout=5)
            return cmd
        except:
            continue
    return None

FFMPEG = find_ffmpeg()
print(f"FFmpeg found: {FFMPEG}")

def reencode_for_browser(src, dst):
    """Re-encode video to H.264 so browsers can play it."""
    if FFMPEG:
        try:
            if os.path.exists(dst):
                os.remove(dst)
            subprocess.run([
                FFMPEG, "-y", "-i", src,
                "-c:v", "libx264", "-preset", "fast",
                "-crf", "23", "-pix_fmt", "yuv420p",
                "-movflags", "+faststart",
                "-an", dst
            ], capture_output=True, timeout=600, check=True)
            return dst
        except:
            pass

    # Fallback: re-encode with OpenCV using avc1
    try:
        cap = cv2.VideoCapture(src)
        fps = cap.get(cv2.CAP_PROP_FPS)
        w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        if os.path.exists(dst):
            os.remove(dst)
        fourcc = cv2.VideoWriter_fourcc(*'avc1')
        writer = cv2.VideoWriter(dst, fourcc, fps, (w, h))
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            writer.write(frame)
        writer.release()
        cap.release()
        if os.path.exists(dst) and os.path.getsize(dst) > 1000:
            return dst
    except:
        pass

    # Last resort: just copy
    shutil.copy2(src, dst)
    return dst

def track_video(file_obj, manual_path, progress=gr.Progress()):
    if manual_path and manual_path.strip() and os.path.exists(manual_path.strip()):
        input_path = manual_path.strip()
    elif file_obj is not None:
        input_path = file_obj if isinstance(file_obj, str) else file_obj.name
    else:
        return None, "Please provide a video path or upload a file."

    if not os.path.exists(input_path):
        return None, f"File not found: {input_path}"

    local_in = os.path.join(ABS_OUTPUTS, "gradio_input.mp4")
    shutil.copy2(input_path, local_in)

    raw_out = os.path.join(ABS_OUTPUTS, "gradio_raw.mp4")
    final_out = os.path.join(ABS_OUTPUTS, "gradio_tracked.mp4")
    CNAMES = {0: 'Player', 1: 'Ball'}
    plist = []

    try:
        vi = sv.VideoInfo.from_video_path(local_in)
        box_a = sv.BoxAnnotator(thickness=6)
        lbl_a = sv.LabelAnnotator(text_scale=1.0, text_thickness=2, text_padding=8)
        trc_a = sv.TraceAnnotator(thickness=4, trace_length=50)

        with sv.VideoSink(raw_out, vi) as sink:
            for fi, frame in enumerate(sv.get_video_frames_generator(local_in)):
                r = demo_model.track(frame, tracker='bytetrack.yaml',
                                     persist=True, conf=0.40, iou=0.5, verbose=False)[0]
                dets = sv.Detections.from_ultralytics(r)

                if len(dets) > 0:
                    keep = []
                    for i in range(len(dets)):
                        x1, y1, x2, y2 = dets.xyxy[i]
                        bw, bh = x2 - x1, y2 - y1
                        if int(dets.class_id[i]) == 0 and bh > 0 and bh / max(bw, 1) > 5.5:
                            continue
                        keep.append(i)
                    if keep:
                        dets = dets[keep]

                labels = []
                for i in range(len(dets)):
                    tid = int(dets.tracker_id[i]) if dets.tracker_id is not None else -1
                    cn = CNAMES.get(int(dets.class_id[i]), "?")
                    cf = float(dets.confidence[i])
                    labels.append(f"#{tid} {cn} {cf:.2f}")

                ann = frame.copy()
                if dets.tracker_id is not None and len(dets) > 0:
                    ann = trc_a.annotate(scene=ann, detections=dets)
                ann = box_a.annotate(scene=ann, detections=dets)
                ann = lbl_a.annotate(scene=ann, detections=dets, labels=labels)

                np_ = sum(1 for c in dets.class_id if c == 0)
                plist.append(np_)
                cv2.rectangle(ann, (0, 0), (620, 42), (0, 0, 0), -1)
                cv2.putText(ann, f"Frame:{fi} | Players:{np_} | YOLOv11m+ByteTrack",
                            (8, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2)
                sink.write_frame(ann)
                progress(fi / max(vi.total_frames, 1))

        # ── KEY FIX: Re-encode to H.264 for browser playback ──
        progress(0.99)
        result_path = reencode_for_browser(raw_out, final_out)

        avg = np.mean(plist) if plist else 0
        return result_path, f"Done! {vi.total_frames} frames. Avg players: {avg:.1f}"

    except Exception as e:
        return None, f"Error: {e}"

# ── UI ──
with gr.Blocks(title="Basketball Tracking", theme=gr.themes.Soft()) as demo:
    gr.Markdown(f"# 🏀 Basketball Tracking — Local Demo\nModel: `{m_path}`")
    with gr.Row():
        with gr.Column():
            manual_path = gr.Textbox(
                label="📁 Video File Path",
                value=INPUT_VIDEO if INPUT_VIDEO else ""
            )
            file_upload = gr.File(label="OR Upload Video", file_types=[".mp4",".avi",".mov"])
            btn = gr.Button("🏃 Start Tracking", variant="primary", size="lg")
            log = gr.Textbox(label="Status", interactive=False, lines=2)
        with gr.Column():
            vout = gr.Video(label="Tracked Result", autoplay=True)

    btn.click(track_video, [file_upload, manual_path], [vout, log])

demo.queue()
print("Launching at http://localhost:7860")
demo.launch(share=False, inbrowser=True, show_error=True,
            allowed_paths=[ABS_OUTPUTS])


---
## 📋 CELL 18 — Final Project Summary Report

### 🎯 PURPOSE
Prints a complete, formatted summary of everything the project did — results, algorithms, improvements, and output locations. **Copy this output into your project report or submission documentation.**

### 📊 What Gets Printed
| Section | Contents |
|---|---|
| Project directory | Full absolute path to BASE_DIR |
| Detection model | YOLOv11m — pretrained source, approach, reason |
| Tracking algorithm | ByteTrack — mechanism, why chosen, performance |
| Accuracy improvements | All 4 changes that reduce false detections |
| Detection results | mAP@50, mAP@50-95, Precision, Recall, F1 (from Cell 9) |
| Tracking results | Unique tracks, avg track length, short tracks (from Cell 13) |

### 🎓 Interview-Ready Summary (Two Lines)
**Detection:** *"YOLOv11m fine-tuned from COCO pretrained weights on ~5000 labeled basketball images from 3 Roboflow datasets. Chosen for real-time speed (30-100 FPS) vs Faster R-CNN (5-10 FPS), with 22% fewer parameters than YOLOv8 and higher mAP."*

**Tracking:** *"ByteTrack with Kalman filter motion model. Chosen because its two-stage matching uses low-confidence detections to recover player tracks during occlusion. Achieves MOTA=77.3% with only 558 ID switches vs 1000+ for SORT, at 171 FPS. SORT loses IDs when players cross paths; DeepSORT is confused by identical team jerseys."*

### ✅ Project Complete Checklist
- [ ] Dataset downloaded (3 Roboflow sources)
- [ ] Quality filter removed blurry/dark/corrupted images
- [ ] Images preprocessed to 640×640
- [ ] Datasets merged with unified player+ball classes
- [ ] YOLOv11m fine-tuned (mAP@50 target: > 0.85)
- [ ] Detection evaluated (mAP, Precision, Recall, F1)
- [ ] ByteTrack tracking on full game video
- [ ] 6-panel tracking analytics dashboard
- [ ] 6-panel spatial analytics + heatmaps
- [ ] Gradio live demo working
- [ ] All outputs saved to OUTPUTS_DIR


In [ ]:
print('='*65)
print('  BASKETBALL DETECTION & TRACKING — LOCAL REPORT')
print('='*65)
print(f'  Project directory: {os.path.abspath(BASE_DIR)}')
print()
print('DETECTION:  YOLOv11m (COCO pretrained, fine-tuned)')
print('TRACKING:   ByteTrack (Kalman + two-stage matching)')
print()
print('ACCURACY IMPROVEMENTS:')
print('  - player+ball only (removed referee)')
print('  - Aspect ratio filter (posts/billboards rejected)')
print('  - 5 datasets merged (~5000+ images)')
print('  - Confidence threshold 0.40')
if 'EVAL_RESULTS' in dir():
    print()
    print('RESULTS:')
    for k,v in EVAL_RESULTS.items(): print(f'  {k}: {v:.4f}')
print('='*65)